<a href="https://colab.research.google.com/github/UFPAJoaoVictorIA/ATIVIDADE-EXTRA-4-Claudomiro/blob/main/AtividadeExtra4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Projeto de Sistema de Memória

Vamos projetar um sistema de memória com as seguintes especificações:
- 16 linhas de dados (BD)
- Pinos de endereço menores que 14
- Capacidade de 32 Gbits

In [1]:
# Parâmetros fornecidos
data_lines = 16  # Linhas de dados (BD)
memory_capacity_gbit = 32  # Capacidade total em Gbits
max_address_pins = 14  # Número máximo de pinos de endereço

print(f"Linhas de Dados (BD): {data_lines}")
print(f"Capacidade da Memória: {memory_capacity_gbit} Gbits")
print(f"Pinos de Endereço Máximos: < {max_address_pins}")

Linhas de Dados (BD): 16
Capacidade da Memória: 32 Gbits
Pinos de Endereço Máximos: < 14


### 1. Capacidade Total da Memória em Bits e Bytes
Primeiro, vamos converter a capacidade total para bits e depois para bytes para facilitar os cálculos.

In [2]:
# Conversão da capacidade total
memory_capacity_bits = memory_capacity_gbit * (1024**3) # Gbits para bits
memory_capacity_bytes = memory_capacity_bits / 8 # Bits para Bytes

print(f"Capacidade da Memória (bits): {memory_capacity_bits} bits")
print(f"Capacidade da Memória (bytes): {memory_capacity_bytes} bytes")

Capacidade da Memória (bits): 34359738368 bits
Capacidade da Memória (bytes): 4294967296.0 bytes


### 2. Número de Palavras e Pinos de Endereço Necessários
Com 16 linhas de dados (BD), cada palavra de memória terá 16 bits (2 bytes).
Vamos calcular o número total de palavras e, a partir disso, o número de pinos de endereço necessários para endereçar cada palavra.

In [3]:
import math

bits_per_word = data_lines # Cada palavra tem 16 bits
bytes_per_word = bits_per_word / 8 # Cada palavra tem 2 bytes

# Número total de palavras que a memória pode armazenar
total_words = memory_capacity_bytes / bytes_per_word

# Número de pinos de endereço necessários para endereçar todas as palavras
address_pins_needed = math.ceil(math.log2(total_words))

print(f"Bits por palavra: {bits_per_word}")
print(f"Bytes por palavra: {bytes_per_word}")
print(f"Total de palavras: {total_words}")
print(f"Pinos de endereço necessários: {address_pins_needed}")

Bits por palavra: 16
Bytes por palavra: 2.0
Total de palavras: 2147483648.0
Pinos de endereço necessários: 31


### 3. Análise da Restrição de Pinos de Endereço
Temos a restrição de que os pinos de endereço devem ser menores que 14. O cálculo acima mostra que precisamos de 34 pinos de endereço (`address_pins_needed = 34`). Isso viola a restrição.

In [4]:
if address_pins_needed < max_address_pins:
    print(f"O número de pinos de endereço ({address_pins_needed}) atende à restrição (< {max_address_pins}).")
else:
    print(f"O número de pinos de endereço ({address_pins_needed}) NÃO atende à restrição (< {max_address_pins}).")

O número de pinos de endereço (31) NÃO atende à restrição (< 14).


### 4. Justificativa e Solução para a Restrição

**Justificativa:**
Se tivéssemos um único chip de memória, ele exigiria 34 pinos de endereço para cobrir os 32 Gbits de capacidade com 16 bits de dados por palavra. Como a restrição é ter menos de 14 pinos de endereço, um único chip não é viável.

**Solução:**
Para atender à restrição de menos de 14 pinos de endereço por chip, precisamos usar múltiplos chips de memória organizados em um arranjo matricial (largura e profundidade).

Vamos supor que usaremos um chip com o número máximo de pinos de endereço permitido, que é 13 (já que deve ser menor que 14). Com 13 pinos de endereço, um chip pode endereçar `2^13` palavras.

`2^13 = 8192` palavras por chip.

In [5]:
address_pins_per_chip = 13 # Usamos 13 pinos de endereço para atender à restrição (<14)
words_per_chip = 2**address_pins_per_chip

print(f"Pinos de endereço por chip (escolhido): {address_pins_per_chip}")
print(f"Palavras por chip: {words_per_chip}")

Pinos de endereço por chip (escolhido): 13
Palavras por chip: 8192


#### Capacidade de um único chip
Com 16 linhas de dados e 8192 palavras, a capacidade de um único chip seria:
`8192 palavras * 16 bits/palavra = 131072 bits = 128 Kbits`

In [6]:
capacity_per_chip_bits = words_per_chip * bits_per_word
capacity_per_chip_kbits = capacity_per_chip_bits / 1024

print(f"Capacidade de um chip: {capacity_per_chip_bits} bits ({capacity_per_chip_kbits} Kbits)")

Capacidade de um chip: 131072 bits (128.0 Kbits)


#### Número de Chips Necessários
Para atingir a capacidade total de 32 Gbits, com cada chip fornecendo 128 Kbits:

`Número de chips = Capacidade Total / Capacidade por Chip`

In [7]:
total_chips_needed = memory_capacity_bits / capacity_per_chip_bits

print(f"Número total de chips necessários: {total_chips_needed} chips")

Número total de chips necessários: 262144.0 chips


Isso significa que precisaríamos de 262.144 chips, o que é um número bastante grande, mas é a consequência de atender à restrição de pinos de endereço por chip.

### 5. Organização do Barramento de Dados
Como temos 16 linhas de dados (BD), cada chip individualmente forneceria 16 bits de dados. Não precisamos de uma organização em largura (para aumentar a largura do barramento de dados) se cada chip já oferece os 16 bits.

### 6. Organização do Barramento de Endereços
Para endereçar 262.144 chips, precisaremos de pinos de seleção de chip (Chip Select - CS). O número de pinos CS necessários é `log2(número total de chips)`.

In [8]:
chip_select_pins = math.ceil(math.log2(total_chips_needed))

print(f"Pinos de seleção de chip (CS) necessários: {chip_select_pins}")

Pinos de seleção de chip (CS) necessários: 18


O barramento de endereços total para o sistema terá:
- 13 pinos de endereço (`A0` a `A12`) compartilhados por todos os chips.
- 18 pinos de seleção de chip (`CS0` a `CS17`) que serão decodificados para ativar o chip correto.

### 7. Desenho Conceitual do Chip e Barramento de Endereços

**Chip Individual:**
Cada chip individual terá:
- **Pinos de Endereço (A0-A12):** 13 pinos para selecionar uma palavra dentro do chip.
- **Pinos de Dados (BD0-BD15):** 16 pinos para entrada/saída de dados.
- **Pino Chip Select (CS):** 1 pino para ativar/desativar o chip. (Este pino será conectado à saída de um decodificador externo).
- **Pino Read/Write (R/W):** 1 pino para controle de leitura/escrita.
- **Pinos de Alimentação (Vcc, GND):** 2 pinos para energia.

**Sistema de Memória (Barramento de Endereços):**
O sistema de memória será composto por:
- **Barramento de Endereços (Total):** O barramento de endereços *externo* (do processador, por exemplo) terá **34 bits** (calculados inicialmente como `address_pins_needed = 34`).
  - Os **13 bits menos significativos** (A0-A12) serão conectados em paralelo a todos os pinos de endereço dos chips.
  - Os **18 bits mais significativos** (A13-A30) serão conectados a um **decodificador de endereço** (e.g., 3-para-8 ou 4-para-16 múltiplos, cascateados) que gerará as 262.144 linhas de seleção de chip (CS). Cada linha de CS do decodificador ativará um grupo específico de chips. *Correção: são 18 bits de seleção de chip, então o decodificador precisaria de 18 entradas e 2^18 saídas.* Isso significa que os bits A13-A30 (total de 18 bits) serão usados para selecionar qual dos 262.144 chips será ativado.
- **Barramento de Dados:** O barramento de dados terá **16 bits** (BD0-BD15). Todos os pinos de dados dos chips estarão conectados em paralelo a este barramento.

**Qual é o tamanho do barramento de endereços?**
O tamanho do barramento de endereços para todo o sistema de memória é de **34 bits**.

*Observação:* A restrição de pinos de endereço "menores que 14" geralmente se refere aos pinos de endereço *de cada chip individual*, e não do barramento de endereços total do sistema. Se a restrição fosse para o sistema inteiro, o projeto seria impossível com 32 Gbits de capacidade e essa largura de barramento de dados.